# 04 Loss of the organisation with acquisition quality
The same analysis after degrading each subject's data: fewer directions, larger voxels, clinical-grade protocols. Classical methods only (networks: notebook 06).

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
subjects = PL.make_subject_lists(cfg); test = [PL.session(sid) for sid in subjects["test"]]
protocols = [protocol for protocol in cfg.run_protocols if protocol != "REF"]; clinical = [protocol for protocol in cfg.clinical_protocols if protocol in protocols]
combinations = [(protocol, "csd") for protocol in protocols]                                      # CSD everywhere
combinations += [(protocol, method) for protocol in clinical for method in ("rumba", "csd_mppca", "csd_interp")]   # slower baselines: clinical protocols only
combinations += [("REF", "msmt_csd+prob")] + [(protocol, "csd+prob") for protocol in clinical]      # same fODFs, probabilistic tracking instead of PFT
combinations += [("REF", "rumba")]
with Timer(cfg, "04_protocol_map"):
    scores = PL.run_grid(cfg, test, combinations, "protocols", device)
scores.groupby(["protocol", "method"])[["n_pairs", "rsi_ree_all", "atlas_rsi_ree_all", "acc", "peak_err_deg", "wdice_ref", "badj_ref_mm"]].median().round(3)

In [ ]:
reference = pd.read_csv(RES / "h1_reference.csv").query("method == 'msmt_csd'")
rsi_table = pd.concat([reference, scores]).pivot_table(index="method", columns="protocol", values="rsi_ree_all", aggfunc="median")
fig, axis = plt.subplots(figsize=(1.2 * rsi_table.shape[1] + 2, 0.6 * rsi_table.shape[0] + 1.5))
image = axis.imshow(rsi_table.to_numpy(), cmap="viridis"); axis.set_xticks(range(rsi_table.shape[1]), rsi_table.columns); axis.set_yticks(range(rsi_table.shape[0]), rsi_table.index)
for (row, column), value in np.ndenumerate(rsi_table.to_numpy()):
    if np.isfinite(value): axis.text(column, row, f"{value:.2f}", ha="center", va="center", color="w")
fig.colorbar(image, label="median RSI"); fig.savefig(FIG / "protocol_map.png", dpi=150, bbox_inches="tight")